#**Tahap Pelatihan & Evaluasi Model**

## **Langkah 9: Model Training**

####**9.1 Load Dataset Final**

In [ ]:
def load_if_needed(var_name, path, dtype=None):
    if var_name in globals() and globals()[var_name] is not None:
        arr = globals()[var_name]
        print(f"  {var_name:30s} : dari memory  shape={arr.shape}")
        return arr
    print(f"  {var_name:30s} : loading dari disk...")
    arr = np.load(path, allow_pickle=False)
    if dtype:
        arr = arr.astype(dtype)
    print(f"  {var_name:30s} : shape={arr.shape} (loaded)")
    return arr

X_train_final = load_if_needed(
    'X_train_final',
    os.path.join(FINAL_DIR, 'X_train_final.npy'),
    dtype=np.float32
)
y_train_final = load_if_needed(
    'y_train_final',
    os.path.join(FINAL_DIR, 'y_train_final.npy'),
    dtype=np.int32
)
X_test = load_if_needed(
    'X_test',
    os.path.join(FINAL_DIR, 'X_test_final.npy'),
    dtype=np.float32
)
y_test = load_if_needed(
    'y_test',
    os.path.join(FINAL_DIR, 'y_test_final.npy'),
    dtype=np.int32
)

# Load feature names
path_feat = os.path.join(FINAL_DIR, 'feature_names.json')
with open(path_feat, 'r') as f:
    feature_names = json.load(f)

N_CLASSES  = len(LABEL_NAMES)
N_FEATURES = X_train_final.shape[1]

print(f"\nX_train_final : {X_train_final.shape}")
print(f"y_train_final : {y_train_final.shape}")
print(f"X_test        : {X_test.shape}")
print(f"y_test        : {y_test.shape}")
print(f"n_classes     : {N_CLASSES}")
print(f"n_features    : {N_FEATURES}")

# Distribusi ringkas
print("\nDistribusi y_train_final:")
for enc in sorted(LABEL_NAMES.keys()):
    n = int((y_train_final == enc).sum())
    print(f"  {LABEL_NAMES[enc]:13s} (enc={enc}): {n:>12,}")

print("\nLoad selesai.")

  X_train_final                  : loading dari disk...
  X_train_final                  : shape=(20302105, 39) (loaded)
  y_train_final                  : loading dari disk...
  y_train_final                  : shape=(20302105,) (loaded)
  X_test                         : loading dari disk...
  X_test                         : shape=(4279721, 39) (loaded)
  y_test                         : loading dari disk...
  y_test                         : shape=(4279721,) (loaded)

X_train_final : (20302105, 39)
y_train_final : (20302105,)
X_test        : (4279721, 39)
y_test        : (4279721,)
n_classes     : 8
n_features    : 39

Distribusi y_train_final:
  Benign        (enc=0):    1,000,000
  Brute_Force   (enc=1):    1,000,000
  DDoS          (enc=2):   10,083,136
  DoS           (enc=3):    3,255,270
  Mirai         (enc=4):    1,963,699
  Recon         (enc=5):    1,000,000
  Spoofing      (enc=6):    1,000,000
  Web_Based     (enc=7):    1,000,000

Load selesai.


####**9.2 Setup Helper Functions**

In [ ]:
def compute_metrics(y_true, y_pred, model_name):
    """Hitung semua metrics evaluasi dan return sebagai dict."""
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average='macro', zero_division=0)
    rec  = recall_score(y_true, y_pred, average='macro', zero_division=0)
    f1   = f1_score(y_true, y_pred, average='macro', zero_division=0)
    mcc  = matthews_corrcoef(y_true, y_pred)

    # Per-class metrics
    report = classification_report(
        y_true, y_pred,
        target_names=[LABEL_NAMES[i] for i in range(N_CLASSES)],
        output_dict=True,
        zero_division=0
    )

    print(f"\n  {model_name} - Hasil Evaluasi:")
    print(f"    Accuracy        : {acc:.6f}")
    print(f"    Macro Precision : {prec:.6f}")
    print(f"    Macro Recall    : {rec:.6f}")
    print(f"    Macro F1-Score  : {f1:.6f}")
    print(f"    MCC             : {mcc:.6f}")
    print(f"\n  Per-class metrics:")
    print(f"    {'Kelas':15s} {'Precision':>10s} {'Recall':>10s} {'F1':>10s} {'Support':>10s}")
    print(f"    {'-'*57}")
    for enc in range(N_CLASSES):
        name = LABEL_NAMES[enc]
        p = report[name]['precision']
        r = report[name]['recall']
        f = report[name]['f1-score']
        s = int(report[name]['support'])
        print(f"    {name:15s} {p:10.4f} {r:10.4f} {f:10.4f} {s:10,}")

    return {
        'model'            : model_name,
        'accuracy'         : float(acc),
        'macro_precision'  : float(prec),
        'macro_recall'     : float(rec),
        'macro_f1'         : float(f1),
        'mcc'              : float(mcc),
        'per_class'        : {
            LABEL_NAMES[enc]: {
                'precision': float(report[LABEL_NAMES[enc]]['precision']),
                'recall'   : float(report[LABEL_NAMES[enc]]['recall']),
                'f1'       : float(report[LABEL_NAMES[enc]]['f1-score']),
                'support'  : int(report[LABEL_NAMES[enc]]['support'])
            }
            for enc in range(N_CLASSES)
        }
    }

# Dict untuk menyimpan semua hasil
all_results = {}

print("Helper functions siap.")

Helper functions siap.


####**9.3 Random Forest**

In [ ]:
print("Pembersihan memory sebelum RF")

# Hapus variabel besar yang tidak diperlukan RF
vars_to_clear = [
    'X_dominant', 'y_dominant',
    'X_moderate_augmented', 'y_moderate_augmented',
    'X_critical_augmented', 'y_critical_augmented',
    'X_critical', 'y_critical',
    'X_moderate', 'y_moderate',
]
for var in vars_to_clear:
    if var in globals():
        del globals()[var]
        print(f"  Hapus dari memory: {var}")

gc.collect()

Pembersihan memory sebelum RF


0

In [ ]:
RF_PARAMS = {
    'n_estimators'      : 300,
    'max_depth'         : 30,
    'min_samples_split' : 2,
    'min_samples_leaf'  : 1,
    'max_features'      : 'sqrt',
    'bootstrap'         : True,
    'class_weight'      : None,
    'n_jobs'            : -1,
    'random_state'      : 42,
    'verbose'           : 1
}

print("Parameter RF:")
for k, v in RF_PARAMS.items():
    print(f"  {k:25s} : {v}")

print("\nMemulai training RF...")
t0 = time.time()

rf_model = RandomForestClassifier(**RF_PARAMS)
rf_model.fit(X_train_final, y_train_final)

t_rf = time.time() - t0
print(f"Training RF selesai: {t_rf:.2f} detik ({t_rf/60:.2f} menit)")

print("\nPrediksi pada test set...")
y_pred_rf = rf_model.predict(X_test)

result_rf = compute_metrics(y_test, y_pred_rf, "Random Forest")
result_rf['training_time_sec'] = float(t_rf)
all_results['RandomForest'] = result_rf

Parameter RF:
  n_estimators              : 300
  max_depth                 : 30
  min_samples_split         : 2
  min_samples_leaf          : 1
  max_features              : sqrt
  bootstrap                 : True
  class_weight              : None
  n_jobs                    : -1
  random_state              : 42
  verbose                   : 1

Memulai training RF...


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done  34 tasks      | elapsed:  8.9min
[Parallel(n_jobs=-1)]: Done 184 tasks      | elapsed: 43.3min
[Parallel(n_jobs=-1)]: Done 300 out of 300 | elapsed: 70.5min finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.


Training RF selesai: 4229.55 detik (70.49 menit)

Prediksi pada test set...


[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:   11.3s
[Parallel(n_jobs=8)]: Done 184 tasks      | elapsed:   53.6s
[Parallel(n_jobs=8)]: Done 300 out of 300 | elapsed:  1.5min finished



  Random Forest - Hasil Evaluasi:
    Accuracy        : 0.856257
    Macro Precision : 0.878556
    Macro Recall    : 0.666819
    Macro F1-Score  : 0.706555
    MCC             : 0.756030

  Per-class metrics:
    Kelas            Precision     Recall         F1    Support
    ---------------------------------------------------------
    Benign              0.8263     0.8983     0.8608    218,760
    Brute_Force         0.9447     0.2224     0.3601      2,612
    DDoS                0.8534     0.9479     0.8982  2,520,784
    DoS                 0.7550     0.4970     0.5995    813,818
    Mirai               0.9995     0.9977     0.9986    490,925
    Recon               0.7898     0.7718     0.7807    136,894
    Spoofing            0.9534     0.8506     0.8991     90,981
    Web_Based           0.9064     0.1488     0.2556      4,947


In [ ]:
# Save model
path_rf = os.path.join(FRAMEWORK_MODELS_DIR, 'rf_model.pkl')
os.makedirs(FRAMEWORK_MODELS_DIR, exist_ok=True)
with open(path_rf, 'wb') as f:
    pickle.dump(rf_model, f)
print(f"\nModel RF tersimpan: {path_rf}")

# Save predictions
np.save(os.path.join(FRAMEWORK_MODELS_DIR, 'y_pred_rf.npy'), y_pred_rf)


Model RF tersimpan: /content/drive/My Drive/Framework/CICIoT2023/Models/rf_model.pkl


In [ ]:
del rf_model
gc.collect()
print("RF selesai.")

RF selesai.


####**9.4 XGBoost**

In [ ]:
# Hitung scale_pos_weight tidak berlaku untuk multiclass
# Gunakan sample_weight via eval_metric
XGB_PARAMS = {
    'n_estimators'      : 300,
    'max_depth'         : 8,
    'learning_rate'     : 0.1,
    'subsample'         : 0.8,
    'colsample_bytree'  : 0.8,
    'min_child_weight'  : 5,
    'gamma'             : 0,
    'reg_alpha'         : 0.1,
    'reg_lambda'        : 1.0,
    'objective'         : 'multi:softmax',
    'num_class'         : N_CLASSES,
    'eval_metric'       : 'mlogloss',
    'tree_method'       : 'hist',
    'device'            : 'cuda',
    'random_state'      : 42,
    'n_jobs'            : -1,
    'verbosity'         : 1
}

print("Parameter XGBoost:")
for k, v in XGB_PARAMS.items():
    print(f"  {k:25s} : {v}")

print("\nMemulai training XGBoost...")
t0 = time.time()

xgb_model = xgb.XGBClassifier(**XGB_PARAMS)
xgb_model.fit(
    X_train_final, y_train_final,
    eval_set=[(X_test, y_test)],
    verbose=50
)

t_xgb = time.time() - t0
print(f"Training XGBoost selesai: {t_xgb:.2f} detik ({t_xgb/60:.2f} menit)")

print("\nPrediksi pada test set...")
y_pred_xgb = xgb_model.predict(X_test)

result_xgb = compute_metrics(y_test, y_pred_xgb, "XGBoost")
result_xgb['training_time_sec'] = float(t_xgb)
all_results['XGBoost'] = result_xgb

Parameter XGBoost:
  n_estimators              : 300
  max_depth                 : 8
  learning_rate             : 0.1
  subsample                 : 0.8
  colsample_bytree          : 0.8
  min_child_weight          : 5
  gamma                     : 0
  reg_alpha                 : 0.1
  reg_lambda                : 1.0
  objective                 : multi:softmax
  num_class                 : 8
  eval_metric               : mlogloss
  tree_method               : hist
  device                    : cuda
  random_state              : 42
  n_jobs                    : -1
  verbosity                 : 1

Memulai training XGBoost...
[0]	validation_0-mlogloss:1.17148
[50]	validation_0-mlogloss:0.33916
[100]	validation_0-mlogloss:0.31386
[150]	validation_0-mlogloss:0.30741
[200]	validation_0-mlogloss:0.30352
[250]	validation_0-mlogloss:0.30072
[299]	validation_0-mlogloss:0.29867
Training XGBoost selesai: 484.31 detik (8.07 menit)

Prediksi pada test set...

  XGBoost - Hasil Evaluasi:
    Accuracy

In [ ]:
# Save model
path_xgb = os.path.join(FRAMEWORK_MODELS_DIR, 'xgb_model.json')
xgb_model.save_model(path_xgb)
print(f"\nModel XGBoost tersimpan: {path_xgb}")

np.save(os.path.join(FRAMEWORK_MODELS_DIR, 'y_pred_xgb.npy'), y_pred_xgb)


Model XGBoost tersimpan: /content/drive/My Drive/Framework/CICIoT2023/Models/xgb_model.json


In [ ]:
del xgb_model
gc.collect()
print("XGBoost selesai.")

XGBoost selesai.


####**9.5 LightGBM**

In [ ]:
LGB_PARAMS = {
    'n_estimators'      : 500,
    'max_depth'         : 8,
    'learning_rate'     : 0.1,
    'num_leaves'        : 63,
    'min_child_samples' : 50,
    'subsample'         : 0.8,
    'subsample_freq'    : 1,
    'colsample_bytree'  : 0.8,
    'reg_alpha'         : 0.1,
    'reg_lambda'        : 1.0,
    'objective'         : 'multiclass',
    'num_class'         : N_CLASSES,
    'metric'            : 'multi_logloss',
    'class_weight'      : None,
    'device'            : 'cpu',  # Diubah ke cpu untuk menghindari error
    'random_state'      : 42,
    'n_jobs'            : -1,
    'verbose'           : -1
}

print("Parameter LightGBM:")
for k, v in LGB_PARAMS.items():
    print(f"  {k:25s} : {v}")

print("\nMemulai training LightGBM...")
t0 = time.time()

lgb_model = lgb.LGBMClassifier(**LGB_PARAMS)
lgb_model.fit(
    X_train_final, y_train_final,
    eval_set=[(X_test, y_test)],
    callbacks=[
        lgb.early_stopping(stopping_rounds=20, verbose=True),
        lgb.log_evaluation(period=50)
    ]
)

t_lgb = time.time() - t0
print(f"Training LightGBM selesai: {t_lgb:.2f} detik ({t_lgb/60:.2f} menit)")

print("\nPrediksi pada test set...")
y_pred_lgb = lgb_model.predict(X_test)

result_lgb = compute_metrics(y_test, y_pred_lgb, "LightGBM")
result_lgb['training_time_sec'] = float(t_lgb)
all_results['LightGBM'] = result_lgb

Parameter LightGBM:
  n_estimators              : 500
  max_depth                 : 8
  learning_rate             : 0.1
  num_leaves                : 63
  min_child_samples         : 50
  subsample                 : 0.8
  subsample_freq            : 1
  colsample_bytree          : 0.8
  reg_alpha                 : 0.1
  reg_lambda                : 1.0
  objective                 : multiclass
  num_class                 : 8
  metric                    : multi_logloss
  class_weight              : None
  device                    : cpu
  random_state              : 42
  n_jobs                    : -1
  verbose                   : -1

Memulai training LightGBM...
Training until validation scores don't improve for 20 rounds
[50]	valid_0's multi_logloss: 0.317486
[100]	valid_0's multi_logloss: 0.305616
[150]	valid_0's multi_logloss: 0.301075
[200]	valid_0's multi_logloss: 0.298448
[250]	valid_0's multi_logloss: 0.296561
[300]	valid_0's multi_logloss: 0.295414
[350]	valid_0's multi_logloss: 

In [ ]:
# Save model
path_lgb = os.path.join(FRAMEWORK_MODELS_DIR, 'lgb_model.txt')
lgb_model.booster_.save_model(path_lgb)
print(f"\nModel LightGBM tersimpan: {path_lgb}")

np.save(os.path.join(FRAMEWORK_MODELS_DIR, 'y_pred_lgb.npy'), y_pred_lgb)


Model LightGBM tersimpan: /content/drive/My Drive/Framework/CICIoT2023/Models/lgb_model.txt


In [ ]:
del lgb_model
gc.collect()
print("LightGBM selesai.")

LightGBM selesai.


####**9.6 Deep Neural Network (DNN)**

In [ ]:
# Arsitektur DNN
DNN_UNITS      = [512, 256, 128, 64]
DNN_DROPOUT    = 0.2
DNN_LR         = 0.001
DNN_BATCH_SIZE = 4096
DNN_EPOCHS     = 100
DNN_PATIENCE   = 15

print("Arsitektur DNN:")
print(f"  Hidden layers    : {DNN_UNITS}")
print(f"  Dropout rate     : {DNN_DROPOUT}")
print(f"  Learning rate    : {DNN_LR}")
print(f"  Batch size       : {DNN_BATCH_SIZE:,}")
print(f"  Max epochs       : {DNN_EPOCHS}")
print(f"  Early stop pat.  : {DNN_PATIENCE}")

# Build model
def build_dnn(n_features, n_classes, units, dropout, lr):
    inputs = keras.Input(shape=(n_features,), name='input')
    x = inputs
    for i, u in enumerate(units):
        x = layers.Dense(u, name=f'dense_{i+1}')(x)
        x = layers.BatchNormalization(name=f'bn_{i+1}')(x)
        x = layers.Activation('relu', name=f'relu_{i+1}')(x)
        x = layers.Dropout(dropout, name=f'dropout_{i+1}')(x)
    outputs = layers.Dense(n_classes, activation='softmax', name='output')(x)

    model = keras.Model(inputs, outputs, name='DNN_Classifier')
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

tf.random.set_seed(42)
dnn_model = build_dnn(N_FEATURES, N_CLASSES, DNN_UNITS, DNN_DROPOUT, DNN_LR)
dnn_model.summary()

# Hitung class weights untuk imbalanced handling
from sklearn.utils.class_weight import compute_class_weight
class_weights_arr = compute_class_weight(
    class_weight='balanced',
    classes=np.arange(N_CLASSES),
    y=y_train_final
)
class_weight_dict = {i: float(w) for i, w in enumerate(class_weights_arr)}
print(f"\nClass weights:")
for enc, w in class_weight_dict.items():
    print(f"  {LABEL_NAMES[enc]:13s} (enc={enc}): {w:.4f}")

# Callbacks
path_dnn_ckpt = os.path.join(FRAMEWORK_MODELS_DIR, 'dnn_best.keras')
callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=DNN_PATIENCE,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=7,
        min_lr=1e-6,
        verbose=1
    ),
    ModelCheckpoint(
        filepath=path_dnn_ckpt,
        monitor='val_loss',
        save_best_only=True,
        verbose=0
    )
]

print("\nMemulai training DNN...")
t0 = time.time()

history = dnn_model.fit(
    X_train_final, y_train_final,
    validation_data=(X_test, y_test),
    batch_size=DNN_BATCH_SIZE,
    epochs=DNN_EPOCHS,
    callbacks=callbacks,
    class_weight=None,
    verbose=1
)

t_dnn = time.time() - t0
print(f"Training DNN selesai: {t_dnn:.2f} detik ({t_dnn/60:.2f} menit)")
print(f"Epoch terbaik       : {np.argmin(history.history['val_loss']) + 1}")
print(f"Best val_loss       : {min(history.history['val_loss']):.6f}")
print(f"Best val_accuracy   : {max(history.history['val_accuracy']):.6f}")

print("\nPrediksi pada test set...")
y_prob_dnn  = dnn_model.predict(X_test, batch_size=DNN_BATCH_SIZE, verbose=0)
y_pred_dnn  = np.argmax(y_prob_dnn, axis=1).astype(np.int32)

result_dnn = compute_metrics(y_test, y_pred_dnn, "DNN")
result_dnn['training_time_sec'] = float(t_dnn)
result_dnn['best_epoch']        = int(np.argmin(history.history['val_loss']) + 1)
result_dnn['best_val_loss']     = float(min(history.history['val_loss']))
result_dnn['best_val_accuracy'] = float(max(history.history['val_accuracy']))
all_results['DNN'] = result_dnn

Arsitektur DNN:
  Hidden layers    : [512, 256, 128, 64]
  Dropout rate     : 0.2
  Learning rate    : 0.001
  Batch size       : 4,096
  Max epochs       : 100
  Early stop pat.  : 15


Model: "DNN_Classifier"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input (InputLayer)              │ (None, 39)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 512)            │        20,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn_1 (BatchNormalization)       │ (None, 512)            │         2,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ relu_1 (Activation)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn_2 (BatchNormalization)       │ (None, 256)            │         1,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ relu_2 (Activation)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn_3 (BatchNormalization)       │ (None, 128)            │           512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ relu_3 (Activation)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn_4 (BatchNormalization)       │ (None, 64)             │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ relu_4 (Activation)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 8)              │           520 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 197,320 (770.78 KB)

 Trainable params: 195,400 (763.28 KB)

 Non-trainable params: 1,920 (7.50 KB)


Class weights:
  Benign        (enc=0): 2.5378
  Brute_Force   (enc=1): 2.5378
  DDoS          (enc=2): 0.2517
  DoS           (enc=3): 0.7796
  Mirai         (enc=4): 1.2923
  Recon         (enc=5): 2.5378
  Spoofing      (enc=6): 2.5378
  Web_Based     (enc=7): 2.5378

Memulai training DNN...
Epoch 1/100
4957/4957 ━━━━━━━━━━━━━━━━━━━━ 45s 7ms/step - accuracy: 0.8478 - loss: 0.3174 - val_accuracy: 0.8352 - val_loss: 0.3328 - learning_rate: 0.0010
Epoch 2/100
4957/4957 ━━━━━━━━━━━━━━━━━━━━ 28s 5ms/step - accuracy: 0.8614 - loss: 0.2824 - val_accuracy: 0.8379 - val_loss: 0.3260 - learning_rate: 0.0010
Epoch 3/100
4957/4957 ━━━━━━━━━━━━━━━━━━━━ 28s 5ms/step - accuracy: 0.8632 - loss: 0.2781 - val_accuracy: 0.8393 - val_loss: 0.3237 - learning_rate: 0.0010
Epoch 4/100
4957/4957 ━━━━━━━━━━━━━━━━━━━━ 28s 5ms/step - accuracy: 0.8641 - loss: 0.2760 - val_accuracy: 0.8403 - val_loss: 0.3219 - learning_rate: 0.0010
Epoch 5/100
4957/4957 ━━━━━━━━━━━━━━━━━━━━ 28s 5ms/step - accuracy: 0.8646 - lo

In [ ]:
# Save model dan history
dnn_model.save(os.path.join(FRAMEWORK_MODELS_DIR, 'dnn_model_final.keras'))
print(f"\nModel DNN tersimpan: {os.path.join(FRAMEWORK_MODELS_DIR, 'dnn_model_final.keras')}")

history_dict = {k: [float(v) for v in vals] for k, vals in history.history.items()}
with open(os.path.join(FRAMEWORK_MODELS_DIR, 'dnn_history.json'), 'w') as f:
    json.dump(history_dict, f, indent=4)

np.save(os.path.join(FRAMEWORK_MODELS_DIR, 'y_pred_dnn.npy'), y_pred_dnn)
np.save(os.path.join(FRAMEWORK_MODELS_DIR, 'y_prob_dnn.npy'), y_prob_dnn)

In [ ]:
del dnn_model, y_prob_dnn
gc.collect()
print("DNN selesai.")

DNN selesai.


####**9.7 Save Hasil dan Ringkasan**

In [ ]:
# Simpan all_results
path_results = os.path.join(FRAMEWORK_RESULTS_DIR, 'training_results_framework.json')
os.makedirs(FRAMEWORK_RESULTS_DIR, exist_ok=True)
with open(path_results, 'w') as f:
    json.dump(all_results, f, indent=4)
print(f"Semua hasil tersimpan: {path_results}")

# Tabel ringkasan perbandingan antar model
print("\n" + "=" * 60)
print("RINGKASAN PERBANDINGAN CLASSIFIER (Framework)")
print("=" * 60)
print(f"\n{'Model':12s} {'Accuracy':>10s} {'Macro P':>10s} {'Macro R':>10s} "
      f"{'Macro F1':>10s} {'MCC':>8s} {'Time(m)':>8s}")
print("-" * 72)

for model_name, res in all_results.items():
    t_min = res['training_time_sec'] / 60
    print(f"  {model_name:10s} "
          f"{res['accuracy']:10.4f} "
          f"{res['macro_precision']:10.4f} "
          f"{res['macro_recall']:10.4f} "
          f"{res['macro_f1']:10.4f} "
          f"{res['mcc']:8.4f} "
          f"{t_min:8.2f}")

print("-" * 72)

# Tabel per-class untuk setiap model
print("\n\nDetail per-class (Macro F1) per model:")
print(f"\n{'Kelas':15s}", end="")
for model_name in all_results:
    print(f"  {model_name:>10s}", end="")
print()
print("-" * (15 + len(all_results) * 13))

for enc in range(N_CLASSES):
    name = LABEL_NAMES[enc]
    print(f"  {name:13s}", end="")
    for model_name, res in all_results.items():
        f1_val = res['per_class'][name]['f1']
        print(f"  {f1_val:>10.4f}", end="")
    print()

print("Langkah 10 berikutnya: Evaluation & Comparison dengan Baseline")

Semua hasil tersimpan: /content/drive/My Drive/Framework/CICIoT2023/Results/training_results_framework.json

RINGKASAN PERBANDINGAN CLASSIFIER (Framework)

Model          Accuracy    Macro P    Macro R   Macro F1      MCC  Time(m)
------------------------------------------------------------------------
  RandomForest     0.8563     0.8786     0.6668     0.7066   0.7560     0.00
  XGBoost        0.8540     0.8670     0.6755     0.7178   0.7520     0.00
  LightGBM       0.8576     0.8672     0.6881     0.7316   0.7583     0.00
  DNN            0.8468     0.8513     0.6518     0.6885   0.7398     0.00
------------------------------------------------------------------------


Detail per-class (Macro F1) per model:

Kelas            RandomForest     XGBoost    LightGBM         DNN
-------------------------------------------------------------------
  Benign             0.8608      0.8598      0.8656      0.8395
  Brute_Force        0.3601      0.4178      0.4434      0.4129
  DDoS           